# Setup
## Necessary library imports

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from tabulate import tabulate
import numpy as np

# Extract 
## Getting the players, player_stats, teams and plays data from those CSV files

In [2]:
#Get players
file_path = "base_data/players.csv"

players_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get players stats
file_path = "playerStats_data/playerStats_2024_ENG.1.csv"

player_stats_2024_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get teams
file_path = "base_data/teams.csv"

teams_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get plays
file_path = "plays_data/plays_2024_ENG.1.csv"

plays_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

player_stats_2024_df.to_csv("players_stats.csv", sep=",")
# player_stats_2024_df = player_stats_2024_df.sort_values(by="shotsFaced_value", ascending=False)
# print(tabulate(player_stats_2024_df.head(99), headers="keys", tablefmt='psql'))

100%|██████████| 11.3M/11.3M [00:02<00:00, 4.75MB/s]
/home/wtc/Desktop/Helix-Football-Data-App/myenv/lib/python3.13/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (0: nickName) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


100%|██████████| 64.0k/64.0k [00:00<00:00, 141kB/s]


100%|██████████| 508k/508k [00:02<00:00, 237kB/s]


100%|██████████| 11.3M/11.3M [00:04<00:00, 2.49MB/s]


# Transform
## Creating a unified dataframe with combined stats, and calculated metrics

In [3]:
# Drop columns I won't be using in each df

players_df = players_df[[
    "athleteId",
    "fullName",
    "slug",
    "weight",
    "displayHeight",
    "height",
    "age",
    "dateOfBirth",
    "citizenship",
    "positionAbbreviation"
]]

player_stats_2024_df = player_stats_2024_df[[
    "teamId",
    "athleteId",
    "appearances_value",
    "foulsCommitted_value",
    "foulsSuffered_value",
    "yellowCards_value",
    "redCards_value",
    "ownGoals_value",
    "goalAssists_value",
    "offsides_value",
    "shotsOnTarget_value",
    "totalShots_value",
    "totalGoals_value",
    "shotsFaced_value",
    "saves_value",
    "goalsConceded_value"
]]

# Getting English first division team IDs so that I can isolate them in the teams dataframe
prem_teams_ids = player_stats_2024_df["teamId"].unique().tolist()

# Creating a dataframe tha's only English first divisiion teams
prem_teams_df = teams_df[teams_df["teamId"].isin(prem_teams_ids)]

# Merge players.csv and player_stats.csv on athleteId
first_merged_df = players_df.merge(player_stats_2024_df, on="athleteId")

# Merge unified players dataframes with teams dataframe so that team name is added as a column
second_merged_df = first_merged_df.merge(teams_df[["teamId", "displayName"]], on="teamId", how="left")

# Rename displayName to something clearer
second_merged_df = second_merged_df.rename(columns={"displayName":"teamName"})

# print(tabulate(second_merged_df.head(300), headers="keys", tablefmt='psql'))
# print(tabulate(prem_teams_df.head(300), headers="keys", tablefmt='psql'))
# print(second_merged_df.columns)

In [ ]:
# Adding calculated stats columns using numpy

unified_df = second_merged_df.copy()
print(unified_df.columns)

# shot accuracy
unified_df["shot_accuracy"] = np.where (
    unified_df["totalShots_value"] > 0,
    unified_df["shotsOnTarget_value"] / unified_df["totalShots_value"],
    np.nan
)

# conversion rate
unified_df["conversion_rate"] = np.where (
    unified_df["totalShots_value"] > 0,
    unified_df["totalGoals_value"] / unified_df["totalShots_value"],
    np.nan
)

# on target conversion rate
unified_df["on_target_conversion_rate"] = np.where (
    unified_df["shotsOnTarget_value"] > 0,
    unified_df["totalGoals_value"] / unified_df["shotsOnTarget_value"],
    np.nan
)

# save percentage
unified_df["save_percentage"] = np.where (
    (unified_df["saves_value"] + unified_df["goalsConceded_value"] > 0) & (unified_df["positionAbbreviation"]=="G"),
    unified_df["saves_value"] / (unified_df["saves_value"]+ unified_df["goalsConceded_value"]),
    np.nan
)

unified_df.head()


Index(['athleteId', 'fullName', 'slug', 'weight', 'displayHeight', 'height',
       'age', 'dateOfBirth', 'citizenship', 'positionAbbreviation', 'teamId',
       'appearances_value', 'foulsCommitted_value', 'foulsSuffered_value',
       'yellowCards_value', 'redCards_value', 'ownGoals_value',
       'goalAssists_value', 'offsides_value', 'shotsOnTarget_value',
       'totalShots_value', 'totalGoals_value', 'shotsFaced_value',
       'saves_value', 'goalsConceded_value', 'teamName'],
      dtype='str')


,athleteId,fullName,slug,weight,displayHeight,height,age,dateOfBirth,citizenship,positionAbbreviation,...,shotsOnTarget_value,totalShots_value,totalGoals_value,shotsFaced_value,saves_value,goalsConceded_value,teamName,shot_accuracy,conversion_rate,save_percentage
0,4946,Michael Keane,michael-keane,181.0,"6' 3""",75.0,32.0,1993-01-11T08:00Z,England,D,...,3,9,3,0,8,20,Everton,0.333333,0.333333,NaN
1,6327,Ben Davies,ben-davies,170.0,"5' 11""",71.0,32.0,1993-04-24T07:00Z,Wales,D,...,1,5,0,0,20,23,Tottenham Hotspur,0.200000,NaN,NaN
2,7441,Mark Travers,mark-travers,181.0,"6' 3""",75.0,26.0,1999-05-18T07:00Z,Republic of Ireland,G,...,0,0,0,77,20,5,AFC Bournemouth,NaN,NaN,0.8
3,9426,Ben Johnson,ben-johnson,148.0,"5' 9""",69.0,26.0,2000-01-24T08:00Z,England,D,...,3,6,1,0,5,38,Ipswich Town,0.500000,0.166667,NaN
4,17828,Adam Webster,adam-webster,163.0,"6' 3""",75.0,30.0,1995-01-04T08:00Z,England,D,...,1,5,0,0,9,11,Brighton & Hove Albion,0.200000,NaN,NaN
